# LLM Quality Monitoring with CloudWatch and Grafana

This notebook demonstrates end-to-end LLM quality monitoring for SageMaker Inference Components:

## Workflow Overview
1. **Invoke SageMaker Inference Components** - Send requests to deployed LLMs
2. **Log to CloudWatch Logs** - Capture requests, responses, and metadata
3. **Calculate Quality Scores** - Use LLM-as-judge (Bedrock Claude) to evaluate responses
4. **Publish Custom Metrics** - Send scores to CloudWatch Metrics
5. **Create Grafana Dashboards** - Visualize quality metrics over time
6. **Setup Grafana Alerts** - Get notified when quality degrades

## Quality Metrics Tracked
- **Safety**: Content safety and policy compliance
- **Relevance**: How well answers address the query
- **Coherence**: Logical flow and consistency
- **Professional Tone**: Appropriate communication style
- **No Harmful Advice**: Absence of dangerous recommendations
- **Latency**: Response time tracking

## Prerequisites
- Active SageMaker Endpoint with Inference Components
- Bedrock access for LLM-as-judge evaluation
- CloudWatch permissions
- Grafana workspace (created if not exists)

## Configuration

Edit the single cell below to set your deployment-specific values. Everything else
in the notebook references these variables — you should not need to edit any other cell.

| Variable | Description |
|---|---|
| `ENDPOINT_NAME` | Your SageMaker endpoint name |
| `INFERENCE_COMPONENT_1` | First inference component name |
| `INFERENCE_COMPONENT_1_LABEL` | Friendly label for the first IC |
| `INFERENCE_COMPONENT_2` | Second inference component name |
| `INFERENCE_COMPONENT_2_LABEL` | Friendly label for the second IC |
| `AWS_ROLE_ARN` | SageMaker execution role ARN (auto-detected in Studio) |
| `MLFLOW_TRACKING_APP_URI` | MLflow 3.4+ tracking server / app ARN |

`REGION` and `ACCOUNT_ID` are auto-detected from your environment.

**Note on trust policy:** The notebook updates `AWS_ROLE_ARN`'s trust policy to allow
self-assumption (required for MLflow's Bedrock scorer).

**Note on MLflow version:** The tracking server/app must be on **MLflow 3.4+**.

## Step 1: Setup and Configuration


In [ ]:
# Install required dependencies
!pip install boto3 requests mlflow==3.8.1 --quiet --upgrade


In [ ]:
# ===================== EDIT THESE VALUES =====================

ENDPOINT_NAME               = "<your-sagemaker-endpoint-name>"
INFERENCE_COMPONENT_1       = "<your-inference-component-1-name>"
INFERENCE_COMPONENT_1_LABEL = "<model-1-label>"
INFERENCE_COMPONENT_2       = "<your-inference-component-2-name>"
INFERENCE_COMPONENT_2_LABEL = "<model-2-label>"
AWS_ROLE_ARN                = "arn:aws:iam::<account-id>:role/service-role/<role-name>"
MLFLOW_TRACKING_APP_URI     = "arn:aws:sagemaker:<region>:<account-id>:mlflow-app/<app-id>"

# ==============================================================

In [ ]:
import boto3
import json
import os
import time
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Any
import mlflow

# ── Auto-detect region & account ──
REGION = boto3.Session().region_name
ACCOUNT_ID = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]

# ── Build the inference-components list from the flat variables above ──
INFERENCE_COMPONENTS = [
    {"name": INFERENCE_COMPONENT_1, "label": INFERENCE_COMPONENT_1_LABEL},
    {"name": INFERENCE_COMPONENT_2, "label": INFERENCE_COMPONENT_2_LABEL},
]

# ── Expose role ARN to MLflow's Bedrock scorer ──
os.environ["AWS_ROLE_ARN"] = AWS_ROLE_ARN

# ── AWS clients (created once, reused everywhere) ──
sm_runtime       = boto3.client("sagemaker-runtime", region_name=REGION)
cloudwatch_logs  = boto3.client("logs",              region_name=REGION)
cloudwatch_metrics = boto3.client("cloudwatch",      region_name=REGION)
bedrock_runtime  = boto3.client("bedrock-runtime",   region_name=REGION)
iam              = boto3.client("iam")
grafana          = boto3.client("grafana",           region_name=REGION)

# Back-compat aliases used by some cells
region = REGION
account_id = ACCOUNT_ID

# ── Fixed / derived configuration ──
LOG_GROUP_PREFIX           = "/aws/sagemaker/inference-quality"
METRIC_NAMESPACE           = "SageMaker/InferenceQuality"
MLFLOW_EVALUATION_MODEL_ID = "bedrock:/global.anthropic.claude-sonnet-4-20250514-v1:0"
MLFLOW_EVALUATION_MODEL_PARAM = {
    "temperature": 0, "max_tokens": 512,
    "anthropic_version": "bedrock-2023-05-31",
    "top_p": 0.9, "stop_sequences": ["}"],
}
THRESHOLDS = {
    "safety_min": 0.8, "relevance_min": 0.7,
    "coherence_min": 0.75, "latency_max_ms": 5000, "composite_min": 0.75,
}
GRAFANA_WORKSPACE_NAME  = "llm-quality-monitoring"
GRAFANA_ROLE_NAME       = "AmazonGrafanaCloudWatchRole"
GRAFANA_DATASOURCE_NAME = "CloudWatch-Quality"
GRAFANA_DASHBOARD_UID   = "llm-quality-monitoring"
GRAFANA_DASHBOARD_TITLE = "LLM Quality Monitoring - SageMaker"

# ── MLflow tracking ──
mlflow.set_tracking_uri(MLFLOW_TRACKING_APP_URI)
mlflow.set_experiment("default")

# ── Summary ──
print(f"✅ Region: {REGION}  Account: {ACCOUNT_ID}  MLflow: {mlflow.__version__}")
print(f"   Endpoint: {ENDPOINT_NAME}")
for ic in INFERENCE_COMPONENTS:
    print(f"   IC: {ic['label']} → {ic['name']}")
print(f"   Role: {AWS_ROLE_ARN}")
print(f"   Tracking: {MLFLOW_TRACKING_APP_URI}")

### Update the role's trust policy

MLflow's Bedrock scorer assumes `AWS_ROLE_ARN` at evaluation time, so the role
needs to trust itself as a principal. This cell updates the trust policy to
add that statement (keeping SageMaker as a trusted service).


In [ ]:
# Update the trust policy on AWS_ROLE_ARN so MLflow's Bedrock scorer can self-assume.
role_name = AWS_ROLE_ARN.split("/")[-1]

new_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole",
        },
        {
            "Effect": "Allow",
            "Principal": {"AWS": AWS_ROLE_ARN},  # self-assumption for MLflow scorers
            "Action": "sts:AssumeRole",
        },
    ],
}

try:
    iam.update_assume_role_policy(
        RoleName=role_name,
        PolicyDocument=json.dumps(new_trust_policy),
    )
    print(f"✅ Trust policy updated for role '{role_name}'.")
except Exception as e:
    print(f"⚠️  Failed to update trust policy for role '{role_name}': {e}")


## Step 2: CloudWatch Logs Setup

Create log groups for each inference component to store request/response data.

In [ ]:
def create_log_group(ic_name: str) -> str:
    """Create CloudWatch log group for inference component."""
    log_group_name = f"{LOG_GROUP_PREFIX}/{ic_name}"
    
    try:
        cloudwatch_logs.create_log_group(logGroupName=log_group_name)
        print(f"✅ Created log group: {log_group_name}")
    except cloudwatch_logs.exceptions.ResourceAlreadyExistsException:
        print(f"ℹ️  Log group exists: {log_group_name}")
    
    # Set retention to 7 days
    try:
        cloudwatch_logs.put_retention_policy(
            logGroupName=log_group_name,
            retentionInDays=7
        )
    except Exception as e:
        print(f"⚠️  Could not set retention: {e}")
    
    return log_group_name

# Create log groups
log_groups = {}
for ic in INFERENCE_COMPONENTS:
    log_groups[ic['name']] = create_log_group(ic['name'])

print(f"\n✅ CloudWatch Logs setup complete!")

## Step 3: Inference and Logging Functions

Functions to invoke endpoints and log to CloudWatch.

In [ ]:
def invoke_inference_component(prompt: str, ic_name: str, endpoint_name: str) -> Dict[str, Any]:
    """Invoke SageMaker Inference Component and measure latency."""
    payload = {
        'inputs': prompt,
        'parameters': {
            'max_new_tokens': 512,
            'temperature': 0.7,
            'top_p': 0.9,
            'do_sample': True
        }
    }
    
    start_time = time.time()
    
    try:
        response = sm_runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            InferenceComponentName=ic_name,
            ContentType='application/json',
            Body=json.dumps(payload)
        )
        
        latency_ms = (time.time() - start_time) * 1000
        response_body = json.loads(response['Body'].read().decode())
        
        # Extract generated text (format may vary by model)
        if isinstance(response_body, list) and len(response_body) > 0:
            generated_text = response_body[0].get('generated_text', '')
        elif 'generated_text' in response_body:
            generated_text = response_body['generated_text']
        elif 'choices' in response_body:
            generated_text = response_body['choices'][0]['message']['content']
        else:
            generated_text = str(response_body)
        
        return {
            'success': True,
            'prompt': prompt,
            'response': generated_text,
            'latency_ms': latency_ms,
            'timestamp': datetime.utcnow().isoformat(),
            'ic_name': ic_name
        }
    
    except Exception as e:
        return {
            'success': False,
            'prompt': prompt,
            'error': str(e),
            'latency_ms': (time.time() - start_time) * 1000,
            'timestamp': datetime.utcnow().isoformat(),
            'ic_name': ic_name
        }

In [ ]:
def log_to_cloudwatch(log_group_name: str, log_data: Dict[str, Any]):
    """Log inference data to CloudWatch Logs."""
    log_stream_name = datetime.utcnow().strftime('%Y/%m/%d/quality-monitoring')
    
    # Create log stream if it doesn't exist
    try:
        cloudwatch_logs.create_log_stream(
            logGroupName=log_group_name,
            logStreamName=log_stream_name
        )
    except cloudwatch_logs.exceptions.ResourceAlreadyExistsException:
        pass
    
    # Put log event
    try:
        cloudwatch_logs.put_log_events(
            logGroupName=log_group_name,
            logStreamName=log_stream_name,
            logEvents=[
                {
                    'timestamp': int(time.time() * 1000),
                    'message': json.dumps(log_data)
                }
            ]
        )
    except Exception as e:
        print(f"⚠️  Failed to log: {e}")

## Step 4: MLflow Quality Evaluation Scorers

Configure MLflow scorers for LLM-as-judge quality evaluation using Amazon Bedrock.

In [ ]:
import mlflow
from mlflow.genai.scorers import Safety, RelevanceToQuery, Guidelines#, Fluency #, make_judge

# Configure MLflow scorers for quality evaluation
safety_scorer = Safety(
    model=MLFLOW_EVALUATION_MODEL_ID, 
    parameters=MLFLOW_EVALUATION_MODEL_PARAM
)

relevance_scorer = RelevanceToQuery(
    model=MLFLOW_EVALUATION_MODEL_ID, 
    parameters=MLFLOW_EVALUATION_MODEL_PARAM
)

professional_tone_scorer = Guidelines(
    name="professional_tone",
    guidelines="The response must be in a professional tone.",
    model=MLFLOW_EVALUATION_MODEL_ID,
    parameters=MLFLOW_EVALUATION_MODEL_PARAM
)

# Combine all scorers
quality_scorers = [
    safety_scorer,
    relevance_scorer,
    professional_tone_scorer,
]

print("✅ MLflow quality scorers configured:")
print(f"   - Safety (built-in)")
print(f"   - RelevanceToQuery (built-in)")
print(f"   - Professional Tone (Guidelines-based)")

## Step 5: Publish Custom Metrics to CloudWatch

Send quality scores as custom CloudWatch metrics.

In [ ]:
def publish_quality_metrics(ic_name: str, ic_label: str, scores: Dict[str, float], latency_ms: float):
    """Publish quality scores as CloudWatch custom metrics."""
    metric_data = []
    timestamp = datetime.utcnow()
    
    # Common dimensions
    dimensions = [
        {'Name': 'InferenceComponentName', 'Value': ic_name},
        {'Name': 'InferenceComponentLabel', 'Value': ic_label},
        {'Name': 'EndpointName', 'Value': ENDPOINT_NAME}
    ]
    
    # Add quality score metrics
    for metric_name, value in scores.items():
        metric_data.append({
            'MetricName': metric_name,
            'Dimensions': dimensions,
            'Value': value,
            'Timestamp': timestamp,
            'Unit': 'None',
            'StorageResolution': 60  # High-resolution metric (1-minute)
        })
    
    # Add latency metric
    metric_data.append({
        'MetricName': 'quality_evaluation_latency',
        'Dimensions': dimensions,
        'Value': latency_ms,
        'Timestamp': timestamp,
        'Unit': 'Milliseconds',
        'StorageResolution': 60
    })
    
    # Composite quality score (average of all scores)
    composite_score = sum(scores.values()) / len(scores)
    metric_data.append({
        'MetricName': 'composite_quality_score',
        'Dimensions': dimensions,
        'Value': composite_score,
        'Timestamp': timestamp,
        'Unit': 'None',
        'StorageResolution': 60
    })
    
    # Publish metrics in batches (max 1000 per call)
    try:
        cloudwatch_metrics.put_metric_data(
            Namespace=METRIC_NAMESPACE,
            MetricData=metric_data
        )
    except Exception as e:
        print(f"⚠️  Failed to publish metrics: {e}")

print("✅ Metric publishing function ready!")

## Step 6: End-to-End Quality Monitoring Pipeline

Complete pipeline: invoke → log → evaluate → publish metrics.

In [ ]:
def monitor_inference_quality(prompt: str, ic_config: Dict[str, str]) -> Dict[str, Any]:
    """Complete quality monitoring pipeline for a single inference using MLflow evaluation."""
    ic_name = ic_config['name']
    ic_label = ic_config['label']
    
    print(f"\n🔄 Processing: {ic_label}")
    
    # Step 1: Invoke inference component
    print("  1️⃣ Invoking inference component...")
    inference_result = invoke_inference_component(prompt, ic_name, ENDPOINT_NAME)
    
    if not inference_result['success']:
        print(f"  ❌ Inference failed: {inference_result.get('error')}")
        return inference_result
    
    print(f"  ✅ Response received ({inference_result['latency_ms']:.0f}ms)")
    response_text = inference_result['response']
    
    # Step 2: Log request/response to CloudWatch Logs
    print("  2️⃣ Logging to CloudWatch Logs...")
    log_to_cloudwatch(log_groups[ic_name], {
        'type': 'inference',
        'timestamp': inference_result['timestamp'],
        'prompt': prompt,
        'response': response_text,
        'latency_ms': inference_result['latency_ms']
    })
    
    # Step 3: Evaluate quality using MLflow
    print("  3️⃣ Evaluating quality with MLflow scorers...")
    
    try:
        eval_dataset = [
            {
                "inputs": {"question": prompt},
                "outputs": response_text,
            }
        ]
        
        eval_results = mlflow.genai.evaluate(
            data=eval_dataset,
            scorers=quality_scorers,
        )
        
        # Extract scores from evaluation results
        # MLflow returns aggregate metrics like 'safety/mean', 'relevance_to_query/mean', etc.
        mlflow_metrics = eval_results.metrics
        
        # Map MLflow scorer results to CloudWatch metric names (normalize the names)
        quality_scores = {
            'safety_score': float(mlflow_metrics.get('safety/mean', 0)),
            'relevance_score': float(mlflow_metrics.get('relevance_to_query/mean', 0)),
            'professional_tone_score': float(mlflow_metrics.get('professional_tone/mean', 0))
        }
        
        print(f"  📊 Quality Scores:")
        for metric, score in quality_scores.items():
            print(f"     {metric}: {score:.3f}")
    
    except Exception as e:
        error_msg = f"MLflow evaluation failed: {e}"
        print(f"  ❌ {error_msg}")
        # Raise error instead of using fallback scores
        raise RuntimeError(error_msg)
    
    # Step 4: Log quality scores to CloudWatch Logs
    print("  4️⃣ Logging quality scores...")
    log_to_cloudwatch(log_groups[ic_name], {
        'type': 'quality_evaluation',
        'timestamp': datetime.utcnow().isoformat(),
        'scores': mlflow_metrics  # Log original MLflow metrics
    })
    
    # Step 5: Publish metrics to CloudWatch Metrics
    print("  5️⃣ Publishing metrics to CloudWatch...")
    publish_quality_metrics(ic_name, ic_label, quality_scores, inference_result['latency_ms'])
    
    print(f"  ✅ Complete for {ic_label}")
    
    return {
        **inference_result,
        'quality_scores': quality_scores
    }

print("✅ Quality monitoring pipeline ready (with MLflow GenAI evaluation format)!")

## Step 7: Test the Pipeline

Run quality monitoring on test prompts.

In [ ]:
test_prompts = [
    "Explain machine learning in simple terms.",
    "What is the difference between supervised and unsupervised learning?",
    "How do neural networks work?",
    "What are transformers in deep learning?",
    "Explain the concept of gradient descent."
]

print(f"🚀 Running quality monitoring on {len(test_prompts)} prompts...\n")
print(f"   Testing {len(INFERENCE_COMPONENTS)} inference components")
print(f"   Total inferences: {len(test_prompts) * len(INFERENCE_COMPONENTS)}")
print("="*80)

results = []

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*80}")
    print(f"Prompt {i}/{len(test_prompts)}: {prompt[:60]}...")
    print(f"{'='*80}")
    
    for ic_config in INFERENCE_COMPONENTS:
        result = monitor_inference_quality(prompt, ic_config)
        results.append(result)
        
        # Small delay to avoid rate limiting
        time.sleep(2)

print(f"\n\n{'='*80}")
print(f"✅ QUALITY MONITORING COMPLETE")
print(f"{'='*80}")
print(f"   Total inferences: {len(results)}")
print(f"   Successful: {sum(1 for r in results if r['success'])}")
print(f"   Failed: {sum(1 for r in results if not r['success'])}")
print(f"\n📊 Metrics published to CloudWatch namespace: {METRIC_NAMESPACE}")
print(f"📝 Logs available in CloudWatch log groups under: {LOG_GROUP_PREFIX}")


## Step 8: Create Grafana Workspace (if not exists)

Set up AWS Managed Grafana for visualization.

In [ ]:
# Grafana IAM role (service-managed). Uses the iam client defined at the top.

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "grafana.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {"StringEquals": {"aws:SourceAccount": ACCOUNT_ID}},
    }],
}

try:
    iam.create_role(
        RoleName=GRAFANA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
    )
    print(f"✅ Created IAM role: {GRAFANA_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"ℹ️  IAM role exists: {GRAFANA_ROLE_NAME}")

cloudwatch_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": [
            "cloudwatch:DescribeAlarmsForMetric",
            "cloudwatch:DescribeAlarms",
            "cloudwatch:ListMetrics",
            "cloudwatch:GetMetricData",
            "cloudwatch:GetMetricStatistics",
            "logs:DescribeLogGroups",
            "logs:GetLogGroupFields",
            "logs:StartQuery",
            "logs:StopQuery",
            "logs:GetQueryResults",
        ],
        "Resource": "*",
    }],
}

iam.put_role_policy(
    RoleName=GRAFANA_ROLE_NAME,
    PolicyName="CloudWatchReadAccess",
    PolicyDocument=json.dumps(cloudwatch_policy),
)

grafana_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{GRAFANA_ROLE_NAME}"
print(f"✅ IAM role configured: {grafana_role_arn}")


In [ ]:
# Check for an existing Grafana workspace and reuse if found; otherwise create one.
existing = [
    w for w in grafana.list_workspaces()["workspaces"]
    if "sagemaker-monitoring" in w["name"] or GRAFANA_WORKSPACE_NAME in w["name"]
]

if existing:
    workspace_id = existing[0]["id"]
    print(f"ℹ️  Using existing Grafana workspace: {existing[0]['name']}")
    print(f"   Workspace ID: {workspace_id}")
else:
    print("🔄 Creating new Grafana workspace...")
    resp = grafana.create_workspace(
        workspaceName=GRAFANA_WORKSPACE_NAME,
        accountAccessType="CURRENT_ACCOUNT",
        authenticationProviders=["AWS_SSO"],
        permissionType="SERVICE_MANAGED",
        workspaceRoleArn=grafana_role_arn,
        workspaceDataSources=["CLOUDWATCH"],
    )
    workspace_id = resp["workspace"]["id"]
    print(f"   Workspace ID: {workspace_id}")
    print("   Waiting for workspace to become active (5-10 min)...")
    while True:
        status = grafana.describe_workspace(workspaceId=workspace_id)["workspace"]["status"]
        print(f"   Status: {status}")
        if status == "ACTIVE":
            break
        time.sleep(15)

workspace = grafana.describe_workspace(workspaceId=workspace_id)["workspace"]
grafana_url = f"https://{workspace['endpoint']}"

print(f"\n✅ Grafana workspace ready!")
print(f"   URL: {grafana_url}")
print(f"   Workspace ID: {workspace_id}")


## Step 9: Configure Grafana Data Source and Dashboard

In [ ]:
# Create a short-lived admin API key for the workspace.
api_key_resp = grafana.create_workspace_api_key(
    workspaceId=workspace_id,
    keyName=f"quality-dashboard-{int(time.time())}",
    keyRole="ADMIN",
    secondsToLive=7200,
)
api_key = api_key_resp["key"]

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

# Reuse an existing CloudWatch data source if we already created one.
print("🔍 Checking for existing CloudWatch data sources...")
existing_datasources = requests.get(
    f"{grafana_url}/api/datasources",
    headers=headers,
).json()

cloudwatch_datasource = next(
    (
        ds for ds in existing_datasources
        if ds.get("name") == GRAFANA_DATASOURCE_NAME and ds.get("type") == "cloudwatch"
    ),
    None,
)

if cloudwatch_datasource:
    ds_uid = cloudwatch_datasource["uid"]
    print(f"ℹ️  CloudWatch data source already exists: {cloudwatch_datasource['name']}")
    print(f"   Using existing UID: {ds_uid}")
else:
    print("🔄 Creating new CloudWatch data source...")
    datasource_resp = requests.post(
        f"{grafana_url}/api/datasources",
        headers=headers,
        json={
            "name": GRAFANA_DATASOURCE_NAME,
            "type": "cloudwatch",
            "access": "proxy",
            "isDefault": True,
            "jsonData": {
                "authType": "default",
                "defaultRegion": REGION,
            },
        },
    ).json()

    if "datasource" in datasource_resp:
        ds_uid = datasource_resp["datasource"]["uid"]
    elif "uid" in datasource_resp:
        ds_uid = datasource_resp["uid"]
    else:
        raise ValueError(f"Failed to create data source: {datasource_resp}")
    print(f"✅ CloudWatch data source created: {ds_uid}")

print(f"\n✅ CloudWatch data source configured with UID: {ds_uid}")


## Step 10: Create LLM Quality Dashboard Panels

In [ ]:
def create_quality_metric_panel(panel_id: int, title: str, metric_name: str, 
                                 y_pos: int, x_pos: int = 0, width: int = 12) -> dict:
    """Create a Grafana panel for quality metrics."""
    return {
        "id": panel_id,
        "type": "timeseries",
        "title": title,
        "gridPos": {"h": 8, "w": width, "x": x_pos, "y": y_pos},
        "datasource": {"type": "cloudwatch", "uid": "${DS_CLOUDWATCH}"},
        "targets": [
            {
                "refId": f"A_{ic['label']}",
                "namespace": METRIC_NAMESPACE,
                "metricName": metric_name,
                "dimensions": {
                    "InferenceComponentName": [ic['name']],
                    "InferenceComponentLabel": [ic['label']],
                    "EndpointName": [ENDPOINT_NAME]
                },
                "statistic": "Average",
                "period": "60",
                "label": ic['label'],
                "region": region,
                "matchExact": True
            }
            for ic in INFERENCE_COMPONENTS
        ],
        "fieldConfig": {
            "defaults": {
                "custom": {
                    "drawStyle": "line",
                    "lineInterpolation": "smooth",
                    "fillOpacity": 10,
                    "lineWidth": 2,
                    "thresholdsStyle": {"mode": "dashed"}
                },
                "unit": "percentunit",
                "min": 0,
                "max": 1,
                "thresholds": {
                    "mode": "absolute",
                    "steps": [
                        {"color": "red", "value": None},
                        {"color": "yellow", "value": 0.7},
                        {"color": "green", "value": 0.85}
                    ]
                }
            }
        },
        "options": {
            "legend": {"displayMode": "list", "placement": "bottom"},
            "tooltip": {"mode": "multi"}
        }
    }

# Create dashboard panels
panels = []
panel_id = 1

# Row 1: Composite Score and Safety
panels.append(create_quality_metric_panel(
    panel_id, "Composite Quality Score", "composite_quality_score", 0, 0, 12
))
panel_id += 1

panels.append(create_quality_metric_panel(
    panel_id, "Safety Score", "safety_score", 0, 12, 12
))
panel_id += 1

# Row 2: Relevance and Professional Tone
panels.append(create_quality_metric_panel(
    panel_id, "Relevance Score", "relevance_score", 8, 0, 12
))
panel_id += 1

panels.append(create_quality_metric_panel(
    panel_id, "Professional Tone Score", "professional_tone_score", 8, 12, 12
))
panel_id += 1

# Row 3: Latency
latency_panel = {
    "id": panel_id,
    "type": "timeseries",
    "title": "Quality Evaluation Latency",
    "gridPos": {"h": 8, "w": 24, "x": 0, "y": 16},
    "datasource": {"type": "cloudwatch", "uid": "${DS_CLOUDWATCH}"},
    "targets": [
        {
            "refId": f"A_{ic['label']}",
            "namespace": METRIC_NAMESPACE,
            "metricName": "quality_evaluation_latency",
            "dimensions": {
                "InferenceComponentName": [ic['name']],
                "InferenceComponentLabel": [ic['label']],
                "EndpointName": [ENDPOINT_NAME]
            },
            "statistic": "Average",
            "period": "60",
            "label": ic['label'],
            "region": region,
            "matchExact": True
        }
        for ic in INFERENCE_COMPONENTS
    ],
    "fieldConfig": {
        "defaults": {
            "custom": {
                "drawStyle": "line",
                "lineInterpolation": "smooth",
                "fillOpacity": 10,
                "lineWidth": 2
            },
            "unit": "ms"
        }
    },
    "options": {
        "legend": {"displayMode": "list", "placement": "bottom"},
        "tooltip": {"mode": "multi"}
    }
}
panels.append(latency_panel)

print(f"✅ Created {len(panels)} dashboard panels")
print(f"   Panels created for metrics:")
print(f"     - composite_quality_score")
print(f"     - safety_score")
print(f"     - relevance_score")
print(f"     - professional_tone_score")
print(f"     - quality_evaluation_latency")

In [ ]:
# Deploy the dashboard. UID / title come from the shared config cell.
dashboard_payload = {
    "dashboard": {
        "uid": GRAFANA_DASHBOARD_UID,
        "title": GRAFANA_DASHBOARD_TITLE,
        "tags": ["sagemaker", "llm", "quality", "monitoring"],
        "timezone": "browser",
        "refresh": "1m",
        "templating": {
            "list": [{
                "name": "DS_CLOUDWATCH",
                "type": "datasource",
                "query": "cloudwatch",
                "current": {"text": GRAFANA_DATASOURCE_NAME, "value": ds_uid},
                "hide": 0,
                "label": "CloudWatch Data Source",
            }],
        },
        "panels": panels,
    },
    "overwrite": True,
}

dashboard_resp = requests.post(
    f"{grafana_url}/api/dashboards/db",
    headers=headers,
    json=dashboard_payload,
)

dashboard_result = dashboard_resp.json()
dashboard_url = f"{grafana_url}{dashboard_result['url']}"

print(f"\n✅ Dashboard deployed successfully!")
print(f"   Dashboard URL: {dashboard_url}")
print(f"   Status: {dashboard_result['status']}")


## Step 11: Create Grafana Alerts

Set up alerts for quality score thresholds.

In [ ]:
# Alert rules for the three most important metrics. Thresholds come from the
# shared THRESHOLDS dict in the config cell.

safety_alert = {
    "uid": "safety-score-alert",
    "title": "Low Safety Score Alert",
    "condition": "A",
    "data": [
        {
            "refId": "A",
            "queryType": "",
            "relativeTimeRange": {"from": 600, "to": 0},
            "datasourceUid": "${DS_CLOUDWATCH}",
            "model": {
                "namespace": METRIC_NAMESPACE,
                "metricName": "safety_score",
                "dimensions": {},
                "statistic": "Average",
                "period": "60",
            },
        },
        {
            "refId": "B",
            "queryType": "",
            "relativeTimeRange": {"from": 600, "to": 0},
            "datasourceUid": "-100",
            "model": {
                "type": "threshold",
                "expression": "A",
                "conditions": [
                    {"evaluator": {"params": [THRESHOLDS["safety_min"]], "type": "lt"}},
                ],
            },
        },
    ],
    "noDataState": "NoData",
    "execErrState": "Alerting",
    "for": "5m",
    "annotations": {
        "description": f"Safety score has fallen below {THRESHOLDS['safety_min']}",
        "summary": "Low safety score detected",
    },
    "labels": {"severity": "critical", "metric": "safety"},
}

relevance_alert = {
    "uid": "relevance-score-alert",
    "title": "Low Relevance Score Alert",
    "condition": "A",
    "data": [
        {
            "refId": "A",
            "queryType": "",
            "relativeTimeRange": {"from": 600, "to": 0},
            "datasourceUid": "${DS_CLOUDWATCH}",
            "model": {
                "namespace": METRIC_NAMESPACE,
                "metricName": "relevance_score",
                "dimensions": {},
                "statistic": "Average",
                "period": "60",
            },
        },
        {
            "refId": "B",
            "queryType": "",
            "relativeTimeRange": {"from": 600, "to": 0},
            "datasourceUid": "-100",
            "model": {
                "type": "threshold",
                "expression": "A",
                "conditions": [
                    {"evaluator": {"params": [THRESHOLDS["relevance_min"]], "type": "lt"}},
                ],
            },
        },
    ],
    "noDataState": "NoData",
    "execErrState": "Alerting",
    "for": "5m",
    "annotations": {
        "description": f"Relevance score has fallen below {THRESHOLDS['relevance_min']}",
        "summary": "Low relevance score detected",
    },
    "labels": {"severity": "warning", "metric": "relevance"},
}

composite_alert = {
    "uid": "composite-quality-alert",
    "title": "Low Composite Quality Score Alert",
    "condition": "A",
    "data": [
        {
            "refId": "A",
            "queryType": "",
            "relativeTimeRange": {"from": 600, "to": 0},
            "datasourceUid": "${DS_CLOUDWATCH}",
            "model": {
                "namespace": METRIC_NAMESPACE,
                "metricName": "composite_quality_score",
                "dimensions": {},
                "statistic": "Average",
                "period": "60",
            },
        },
        {
            "refId": "B",
            "queryType": "",
            "relativeTimeRange": {"from": 600, "to": 0},
            "datasourceUid": "-100",
            "model": {
                "type": "threshold",
                "expression": "A",
                "conditions": [
                    {"evaluator": {"params": [THRESHOLDS["composite_min"]], "type": "lt"}},
                ],
            },
        },
    ],
    "noDataState": "NoData",
    "execErrState": "Alerting",
    "for": "5m",
    "annotations": {
        "description": f"Overall quality score has fallen below {THRESHOLDS['composite_min']}",
        "summary": "Low overall quality detected",
    },
    "labels": {"severity": "warning", "metric": "composite_quality"},
}

print("✅ Alert rules configured for current metrics:")
print(f"   Safety Score Alert:        threshold < {THRESHOLDS['safety_min']}  (critical)")
print(f"   Relevance Score Alert:     threshold < {THRESHOLDS['relevance_min']}  (warning)")
print(f"   Composite Quality Alert:   threshold < {THRESHOLDS['composite_min']} (warning)")
print(f"\nℹ️  Grafana alert provisioning via API requires additional setup.")
print(f"   Create these alerts manually in the Grafana UI at:")
print(f"   {grafana_url}/alerting/list")


## Step 12: View Results and Summary

In [ ]:
# Summary statistics
successful_results = [r for r in results if r["success"]]

if successful_results:
    print("\n" + "=" * 80)
    print("📊 QUALITY MONITORING SUMMARY")
    print("=" * 80)

    for ic in INFERENCE_COMPONENTS:
        ic_results = [r for r in successful_results if r["ic_name"] == ic["name"]]
        if not ic_results:
            continue

        print(f"\n🔹 {ic['label']} ({ic['name']})")
        print(f"   Total inferences: {len(ic_results)}")

        avg_safety = sum(r["quality_scores"].get("safety_score", 0) for r in ic_results) / len(ic_results)
        avg_relevance = sum(r["quality_scores"].get("relevance_score", 0) for r in ic_results) / len(ic_results)
        avg_professional = sum(r["quality_scores"].get("professional_tone_score", 0) for r in ic_results) / len(ic_results)
        avg_latency = sum(r["latency_ms"] for r in ic_results) / len(ic_results)

        print(f"\n   Average Quality Scores:")
        print(f"     Safety:            {avg_safety:.3f}")
        print(f"     Relevance:         {avg_relevance:.3f}")
        print(f"     Professional Tone: {avg_professional:.3f}")
        print(f"\n   Average Latency: {avg_latency:.0f}ms")

        print(f"\n   Threshold Status:")
        print(f"     {'✅' if avg_safety >= THRESHOLDS['safety_min'] else '❌'} Safety "
              f"({avg_safety:.3f} vs >= {THRESHOLDS['safety_min']})")
        print(f"     {'✅' if avg_relevance >= THRESHOLDS['relevance_min'] else '❌'} Relevance "
              f"({avg_relevance:.3f} vs >= {THRESHOLDS['relevance_min']})")
        print(f"     {'✅' if avg_latency <= THRESHOLDS['latency_max_ms'] else '❌'} Latency "
              f"({avg_latency:.0f}ms vs <= {THRESHOLDS['latency_max_ms']}ms)")

print("\n" + "=" * 80)
print("🎉 SETUP COMPLETE!")
print("=" * 80)
print(f"\n📊 Grafana Dashboard:\n   {dashboard_url}")
print(f"\n📝 CloudWatch Logs:")
for ic in INFERENCE_COMPONENTS:
    print(f"   {ic['label']}: {log_groups[ic['name']]}")
print(f"\n📈 CloudWatch Metrics:")
print(f"   Namespace: {METRIC_NAMESPACE}")
print(f"   Available metrics:")
for m in (
    "safety_score",
    "relevance_score",
    "professional_tone_score",
    "composite_quality_score",
    "quality_evaluation_latency",
):
    print(f"     - {m}")
print(f"\n🔔 Alerting:")
print(f"   Configure alerts in Grafana UI: {grafana_url}/alerting/list")
print("\n" + "=" * 80)


## Next Steps

1. **View Dashboard**: Open the Grafana dashboard URL above to see quality metrics
2. **Configure Alerts**: Set up alert notifications in Grafana (Email, Slack, SNS, etc.)
3. **Continuous Monitoring**: Run this pipeline continuously or on a schedule
4. **Customize Metrics**: Add more evaluation criteria based on your use case
5. **Integration**: Integrate with CI/CD for automated quality checks

### Manual Alert Setup in Grafana:
1. Go to **Alerting → Alert rules** in Grafana
2. Create new alert rule
3. Select CloudWatch data source
4. Choose metric (e.g., `safety_score`)
5. Set condition (e.g., `WHEN avg() IS BELOW 0.8`)
6. Configure notification channel
7. Save alert

### CloudWatch Logs Insights Queries:

**View all inferences:**
```
fields @timestamp, type, latency_ms, prompt, response
| filter type = "inference"
| sort @timestamp desc
```

**View quality scores:**
```
fields @timestamp, scores.`safety/mean`, scores.`relevance_to_query/mean`, scores.`professional_tone/mean`
| filter type = "quality_evaluation"
| sort @timestamp desc
```

**View quality scores with all fields:**
```
fields @timestamp, scores
| filter type = "quality_evaluation"
| sort @timestamp desc
```